# <font size="8">Image Segmentation: Embryo</font>


# Library

In [ ]:
#%pip install tifffile numpy matplotlib scikit-learn scipy pandas plotly
#%pip install ipython nbformat
#!pip install nbformat ipython ipykernel
#!pip install notebook nbconvert

# Importing standard libraries
import tifffile                                            # a library for reading and writing TIFF files, commonly used for handling large image data in scientific applications
import numpy as np             
import matplotlib.pyplot as plt                            # a plotting library used for creating static, interactive and animated visualisations in python
from sklearn.cluster import KMeans                         # implements the K-means clustering algorithm, an unsupervised learning method for grouping data points
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as colors
from scipy import stats
from sklearn.decomposition import PCA
import pandas as pd
import plotly.express as px
import xml.etree.ElementTree as ET
import plotly
import plotly.express as px

import plotly.io as pio
pio.renderers.default = 'vscode'


# Loading and Inspecting the Data (ome.tiff file)

In [ ]:
# Establishing File Path
file_path = '/Users/tolgakiymak/Desktop/kings_masters/Extended_Research_Project/Embryo_Data/MURINE_EMBRYO/MURINE_EMBRYO.ome.tif' 

# Loading the Raw Image Data
img_raw = tifffile.imread(file_path)

# Automatically Extract Channel Names from OME Metadata, if available
with tifffile.TiffFile(file_path) as tif:
    omexml_string = tif.ome_metadata

if omexml_string:
    namespaces = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
    root = ET.fromstring(omexml_string)
    channels = root.findall('.//ome:Channel', namespaces)
    all_channel_names = [c.attrib.get('Name', f"Channel_{i}") for i, c in enumerate(channels)]
else:
    print("Warning: No OME metadata found. Generating placeholder names.")
    all_channel_names = [f"Channel_{i}" for i in range(img_raw.shape[0])]

# Only drop channel 0 (0TIC) — always useless as it is the total ion count
# Se and any other poor channels will be dropped automatically by the quality assessment
img = np.delete(img_raw, [0], axis=0)
channel_names = [name for i, name in enumerate(all_channel_names) if i not in [0]]
num_channels = len(channel_names)

# Setup Colours Safely
base_colours = ['lime', 'blue', 'yellow', 'cyan', 'magenta', 'orange',
                'purple', 'pink', 'teal', 'lightgreen', 'gold', 'white']
qupath_colours = [base_colours[i % len(base_colours)] for i in range(num_channels)]

# Print Checks
print(f"Original Data Shape: {img_raw.shape}")
print(f"Data Shape after dropping 0TIC: {img.shape}")
print(f"Data Type: {img.dtype}")
print(f"Min value: {np.min(img)}")
print(f"Max value: {np.max(img)}\n")
print(f"Successfully loaded {num_channels} channels:")
print(channel_names)

# Loading and Inspecting Meta Data

In [ ]:
# Loading metadata CSV
metadata_path = '/Users/tolgakiymak/Desktop/kings_masters/Extended_Research_Project/Embryo_Data/meta_data/meta_data.csv'
meta_df = pd.read_csv(metadata_path, index_col=0)

# Parse the min/max threshold lists from the metadata
import ast
import re
all_min_thresholds = ast.literal_eval(meta_df.loc['min thresholds', '0'])
all_max_thresholds = ast.literal_eval(meta_df.loc['max thresholds', '0'])
all_channel_names_meta = ast.literal_eval(meta_df.loc['processed isotopes', '0'])

# Build a lookup dict: channel name → (min, max)
# re.sub strips leading numbers from metadata names so '23Na' becomes 'Na', '56Fe' becomes 'Fe' etc.
# This makes them match channel_names_filtered which uses short names
threshold_lookup = {
    re.sub(r'^\d+', '', name): (all_min_thresholds[i], all_max_thresholds[i])
    for i, name in enumerate(all_channel_names_meta)
}

print("Threshold lookup built:")
for name, (mn, mx) in threshold_lookup.items():
    print(f"  {name}: min={mn}, max={mx}")

# Channel Quality Assessment 

In [ ]:
# For each channel we calculate 4 metrics that together indicate data quality
# A good channel has: high SNR (signal to noise ratio), low noise, sufficient tissue coverage, and meaningful signal

quality_metrics = []

for i, name in enumerate(channel_names):
    channel_data = img[i].flatten()
    tissue_pixels = channel_data[channel_data > 0]

    if len(tissue_pixels) == 0:
        quality_metrics.append({
            'Channel': name,
            'SNR': 0,
            'CV': 999,
            'Tissue_Coverage_%': 0,
            'Mean_Intensity': 0,
            'Noise_Floor': 0,
            'Flag': 'EMPTY'
        })
        continue

    # SIGNAL TO NOISE RATIO (SNR)
    # Mean signal divided by standard deviation
    # High SNR = strong consistent signal relative to noise
    # Low SNR = signal is buried in noise (like Selenium)
    mean_signal = np.mean(tissue_pixels)
    std_noise = np.std(tissue_pixels)
    snr = mean_signal / (std_noise + 1e-10)

    # COEFFICIENT OF VARIATION (CV)
    # std / mean — measures how noisy the channel is relative to its signal
    # High CV = very noisy, Low CV = consistent signal
    cv = (std_noise / (mean_signal + 1e-10)) * 100

    # TISSUE COVERAGE
    # What percentage of pixels have any signal at all
    # Very low coverage = channel barely detected anything
    tissue_coverage = (len(tissue_pixels) / len(channel_data)) * 100

    # NOISE FLOOR
    # The 5th percentile of tissue pixels
    # If this is very close to the mean, most signal is just noise
    noise_floor = np.percentile(tissue_pixels, 5)

    # Flagging Criteria:
    # A channel is flagged as POOR if it fails two or more criteria
    flags = []
    if snr < 1.0:
        flags.append('low SNR')
    if cv > 200:
        flags.append('high CV')
    if tissue_coverage < 10:
        flags.append('low coverage')
    if noise_floor > mean_signal * 0.8:
        flags.append('high noise floor')

    if len(flags) >= 2:
        flag_status = f'POOR ({", ".join(flags)})'
    elif len(flags) == 1:
        flag_status = f'BORDERLINE ({flags[0]})'
    else:
        flag_status = 'GOOD'

    quality_metrics.append({
        'Channel': name,
        'SNR': round(snr, 3),
        'CV': round(cv, 1),
        'Tissue_Coverage_%': round(tissue_coverage, 1),
        'Mean_Intensity': round(mean_signal, 1),
        'Noise_Floor': round(noise_floor, 1),
        'Flag': flag_status
    })

quality_df = pd.DataFrame(quality_metrics)
print(quality_df.to_string(index=False))


#### OUTPUT EXPLAINED #### 
# The quality assessment table shows 6 metrics for every channel, calculated only on the (tissue) pixels
# SNR (Signal to Noise Ratio): mean signal divided by standard deviation. Above 1.0 = signal dominates noise. Below 1.0 = noise dominates signal (channel is unreliable)
# CV (Coefficient of Variation): standard deviation as a percentage of the mean. Below 200% = acceptable variability. Above 200% = signal is so variable it is likely noise
# Tissue_Coverage_%: percentage of image pixels that have any detectable signal. Below 10% = element was barely present or detector failed
# Mean_Intensity: average brightness of tissue pixels. Useful for comparing relative abundance of elements across the tissue
# Noise_Floor: the 5th percentile of tissue pixel values. If this is close to the mean (above 80%), most signal is a flat noisy baseline rather than real tissue variation
# Flag: GOOD = passed all criteria. BORDERLINE = failed one criterion, review manually. POOR = failed two or more criteria, will be automatically dropped before PCA
# A channel needs to fail at least two criteria to be flagged POOR. This prevents a single unusual metric from incorrectly removing a good channel

# Visualising Channel Quality 

In [ ]:
# Visualising the channel quality 
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colours = ['tomato' if 'POOR' in f else 'orange' if 'BORDERLINE' in f else 'steelblue'
           for f in quality_df['Flag']]

# SNR per channel
axes[0].barh(quality_df['Channel'], quality_df['SNR'], color=colours)
axes[0].axvline(x=1.0, color='red', linestyle='--', label='SNR threshold (1.0)')
axes[0].set_title('Signal to Noise Ratio per Channel', fontsize=12)
axes[0].set_xlabel('SNR')
axes[0].legend()

# CV per channel
axes[1].barh(quality_df['Channel'], quality_df['CV'], color=colours)
axes[1].axvline(x=200, color='red', linestyle='--', label='CV threshold (200%)')
axes[1].set_title('Coefficient of Variation per Channel', fontsize=12)
axes[1].set_xlabel('CV (%)')
axes[1].legend()

# Tissue Coverage per channel
axes[2].barh(quality_df['Channel'], quality_df['Tissue_Coverage_%'], color=colours)
axes[2].axvline(x=10, color='red', linestyle='--', label='Coverage threshold (10%)')
axes[2].set_title('Tissue Coverage per Channel', fontsize=12)
axes[2].set_xlabel('Coverage (%)')
axes[2].legend()

plt.suptitle('Channel Quality Assessment\n(Red = POOR, Orange = BORDERLINE, Blue = GOOD)',
             fontsize=14)
plt.tight_layout()
plt.show()

# Auto/Manually Dropping Poor Channels

In [ ]:
# AUTOMATIC: drops channels flagged as POOR by the quality assessment
# Comment out this block if you want to use manual selection instead
#poor_channels = quality_df[quality_df['Flag'].str.startswith('POOR')]['Channel'].tolist()
#borderline_channels = quality_df[quality_df['Flag'].str.startswith('BORDERLINE')]['Channel'].tolist()
#print(f"POOR channels (will be dropped): {poor_channels}")
#print(f"BORDERLINE channels (review manually): {borderline_channels}")

# OPTION 2 — MANUAL: specify exactly which channels to drop
# Uncomment this block if you want full manual control
poor_channels = []  # start with empty list
manual_drop = ['Mn', 'K']  # ← add channel names here
poor_channels = poor_channels + manual_drop

# OPTIONAL: add extra channels to drop on top of automatic selection
# Only applies when using OPTION 1
#extra_drop = []  # ← add any extra channels here e.g. ['Se']
#poor_channels = poor_channels + extra_drop

# Filtering channels and image
good_channel_mask = [name not in poor_channels for name in channel_names]
channel_names_filtered = [name for name in channel_names if name not in poor_channels]
img_filtered = img[good_channel_mask]

print(f"\nOriginal channels: {len(channel_names)}")
print(f"Channels after dropping POOR: {len(channel_names_filtered)}")
print(f"Kept channels: {channel_names_filtered}")

# Setting Display Ranges from Metadata

In [ ]:
num_channels = img_filtered.shape[0]  # Use the actual dimension instead
print(f"img_filtered shape: {img_filtered.shape}")
print(f"num_channels value: {num_channels}")

In [ ]:
# I am storing the min/max for each channel in this list
channel_ranges = []

# This is the start of the for loop, going through all the channels
# "for i in...": This starts the loop. The first time it runs, i is 0. The next time, i is 1 and so on. i is the index counter
# "1" This is the number of rows (1)
# "num_channels_to_view": number of channels (5)
# "i + 1": The position of the current plot. Matplotlib counts positoons starting at 1, but python starts at 0. So when 1=0, we plot in position
for i in range(num_channels):
    
    # Directly fetching min and max from the metadata threshold lookup
    # If a channel name doesn't match, it will throw a KeyError so you can catch name mismatches early
    optimum_min, optimum_max = threshold_lookup[channel_names_filtered[i]]

    # Storing the result
    channel_ranges.append((optimum_min, optimum_max))

    print(f"Channel {i} ({channel_names_filtered[i]}): Min={optimum_min}, Max={int(optimum_max)}")

# Custom Colourmaps (1 row for quick visualisation)

In [ ]:
plt.figure(figsize=(20,5))

for i in range(num_channels):
  plt.subplot(1, num_channels, i + 1)

  vmin, vmax = channel_ranges[i]
  cmap_name = f"black_to_{qupath_colours[i]}"
  custom_cmap = LinearSegmentedColormap.from_list(cmap_name, ['black', qupath_colours[i]])

  plt.imshow(img_filtered[i], cmap=custom_cmap, vmin=vmin, vmax=vmax, origin='upper')
  plt.title(f"{channel_names_filtered[i]}\nColor: {qupath_colours[i]}", fontsize=8)
  plt.axis('off')

plt.tight_layout()
plt.show

# Comparison between log and linear

In [ ]:
# Channel Settings
channels_to_test = len(channel_names_filtered)

# ← Save suggestions here so other cells can use them
scale_suggestions = {}

fig, axes = plt.subplots(nrows=channels_to_test, ncols=3, figsize=(18, 4 * channels_to_test))

for i in range(channels_to_test):
    
    channel_data = img_filtered[i]
    flat_data = channel_data.flatten()
    valid_data = flat_data[flat_data > 0]

    if len(valid_data) > 0:
        robust_max = np.percentile(valid_data, 99.9)
    else:
        robust_max = 1

    ax_lin = axes[i, 0]
    ax_lin.imshow(channel_data, cmap='inferno', vmax=robust_max, origin='upper')
    ax_lin.set_title(f"{channel_names_filtered[i]} - Linear Scale")
    ax_lin.axis('off')

    ax_log = axes[i, 1]
    # "vmin=1e-10" was changed to "vmin=1", stopping near-zero noise pixels from dragging the scale down
    # "vmax=robust_max" (99.9th percentile) was changed to "vmax=np.percentile(valid_data, 95)", bringing the top of the scale down so the colour range spreads more evenly across the tissue signal, rather than being dominated by a few very bright pixels
    ax_log.imshow(channel_data, cmap='magma',
                  norm=colors.LogNorm(vmin=1, vmax=np.percentile(valid_data, 95)), origin='upper')  
    ax_log.set_title(f"{channel_names_filtered[i]} - Log Scale")
    ax_log.axis('off')

    ax_hist = axes[i, 2]

    if len(valid_data) > 0:
        ax_hist.hist(valid_data, bins=100, color='gray', log=True)
        ax_hist.set_xlabel("Pixel Brightness")
        ax_hist.set_ylabel("Count (Log Scale)")

        skew = stats.skew(valid_data)
        cv = np.std(valid_data) / np.mean(valid_data)
        percentile_ratio = np.percentile(valid_data, 99) / (np.median(valid_data) + 1e-10)

        if skew > 3 and percentile_ratio > 50:
            suggestion = "LOG"
            suggest_color = 'red'
        elif skew > 2 or cv > 2:
            suggestion = "POSSIBLY LOG"
            suggest_color = 'orange'
        else:
            suggestion = "LINEAR"
            suggest_color = 'green'

        # ← Save suggestion for this channel
        scale_suggestions[channel_names_filtered[i]] = suggestion

        ax_hist.set_title(
            f"Distribution  |  Skew: {skew:.1f}  |  CV: {cv:.1f}  |  P99/Median: {percentile_ratio:.0f}x",
            fontsize=8
        )
        ax_hist.text(0.5, 0.9, f"Suggestion: {suggestion}", transform=ax_hist.transAxes,
                     color=suggest_color, weight='bold', ha='center')

plt.tight_layout()
plt.show()

print("\nScale suggestions saved:")
print(scale_suggestions)


#### OUTPUT EXPLAINED ####
# The histogram now shows the actual metric values (skew, CV, P00/median ratio) for every channel
# This means you are not just blindly trusting the suggestions, as you can see the numbers that drove the decision

# The code doesn't traces the tissue
# In the image, img_filtered[i] is a 2D array of shape (height, width) where each value is the elemental intensity measured at that physical location. 


# This cell produces three plots per channel: a linear scale image, a log scale image, and an intensity distribution histogram
# The histogram drives the automatic log/linear suggestion for each channel, saved into scale_suggestions for use in later cells

# LINEAR IMAGE (left column):
# Displays raw pixel intensities mapped directly to colour brightness
# vmax is set to the 99.9th percentile to prevent a few extreme hotspots from making the rest of the tissue appear black
# Uses inferno colormap

# LOG IMAGE (middle column):
# Displays the same data but with logarithmic colour scaling
# vmin=1 prevents near-zero noise pixels from dragging the colour scale down
# vmax=95th percentile brings the top of the scale down so colour spreads more evenly across tissue signal
# Uses magma colormap: black 
# Log scaling is useful when a channel has a few very bright hotspots that compress everything else into darkness

# HISTOGRAM (right column): three metrics drive the log/linear decision:
# Skew: measures how lopsided the distribution is toward bright outliers. Above 3 = heavily right-skewed, most pixels are dim but a few are extremely bright
# CV (Coefficient of Variation): standard deviation divided by mean, expressed as a ratio. Above 2.0 (equivalent to 200%) = signal is highly variable relative to its average
# CV of 200% = signal varies by twice its own mean. Highly variable, suggests either extreme heterogeneity or noise domination
# Percentile Ratio (P99/Median): 99th percentile divided by the median. Above 50x = the brightest 1% of pixels are more than 50 times brighter than the typical pixel

# DECISION LOGIC:
# LOG = skew above 3 AND percentile ratio above 50 (both conditions must be met)
# POSSIBLY LOG = skew above 2 OR CV above 2 (only one condition needed, moderate threshold)
# LINEAR = none of the above conditions met

# All suggestions are saved into scale_suggestions dictionary so downstream Plotly scatter plots
# can automatically display a second log-scaled plot for flagged channels

# <font size="8">PCA Analysis</font>



# Preparing the Data 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

# Reshaping image into a table:
# img shape is (filtered channels, Height, Width)
# We need to reshape it into a 2D table: (pixels, channels)
# Each row = one pixel, each column = one element/channel
n_channels, height, width = img_filtered.shape
n_pixels = height * width

# Better Background Removal:
# Sum all channels per pixel to get a 2D intensity map
total_intensity = img_filtered.reshape(n_channels, -1).T.sum(axis=1).reshape(height, width)

# 10th percentile of non-zero pixels = natural background level threshold
# Stronger than sum > 0 — removes dim near-background pixels too
background_threshold = np.percentile(total_intensity[total_intensity > 0], 10)

# Binary mask: True = tissue pixel, False = background
tissue_mask = total_intensity > background_threshold

# Flatten mask to index into reshaped pixel array
img_reshaped = img_filtered.reshape(n_channels, n_pixels).T  # (pixels, channels)
tissue_indices = np.where(tissue_mask.flatten())[0]

# Build df using only tissue pixels
df = pd.DataFrame(img_reshaped[tissue_indices], columns=channel_names_filtered)

# Freezing tissue_indices here so visualisation can use it later in k-means image segmentation
tissue_indices_final = tissue_indices.copy()

print(f"Total pixels: {n_pixels:,}")
print(f"Tissue pixels after background removal: {len(df):,}")
print(f"Background pixels removed: {n_pixels - len(df):,}")
print(df.describe().round(2))

In [ ]:
# x axis is pixels, y axis is intensity, each line is a channel
# intensity is basically the count of how many ions of that element were detected at that pixel location. 
# Higher intensity = more of that element in that spot on the tissue

df.head()

# Scaling (Z-score normalisation)

In [ ]:
# In Part 1, I used StandardScaler which normalises the data based on mean and std
# However, extreme outlier pixels can heavily skew the mean and std
# In Part 2, I clipped to a percentile range FIRST to remove outliers,
# then z-score normalise using StandardScaler so all channels still have equal weightings

# Step 1: Percentile clipping — removes outliers by capping extreme values
# Step 2: Z-score normalisation — standardises all channels to mean=0, std=1

def clip_and_normalise(data, low_pct, high_pct):
    """
    STEP 1 — Percentile clipping:
    For each channel, any pixel below low_pct or above high_pct is capped
    This removes extreme outliers before any normalisation happens

    STEP 2 — Z-score normalisation:
    After clipping, each channel is normalised to mean=0, std=1
    So no single element dominates PCA just because its numbers are bigger
    """
    # Step 1: Percentile Clipping
    clipped = np.zeros_like(data, dtype=float)
    for ch in range(data.shape[1]):
        low = np.percentile(data[:, ch], low_pct)
        high = np.percentile(data[:, ch], high_pct)
        clipped[:, ch] = np.clip(data[:, ch], low, high)

    # Step 2: Z-score Normalisation
    scaler2 = StandardScaler()
    normalised = scaler2.fit_transform(clipped)

    return normalised

X = df.values

# 99th Percentile Clipping

In [ ]:
# 1-99th Percentile Clipping
# More lenient — removes only the bottom 1% and top 1% of pixel values per channel
# Keeps more of the real signal range with less outlier protection
# Better for preserving biological variation where extreme values may be meaningful

X_scaled_1_99 = clip_and_normalise(X, 1, 99)  

# Checking the stats 
# Converting the NumPy array (X_scaled_1_99) into pandas dataframes with proper channel name labels
dfx_1_99 = pd.DataFrame(data=X_scaled_1_99, columns=channel_names_filtered)


print("\n1-99th Percentile Clipping (mean ~0, std ~1):")
print(dfx_1_99.describe().round(2))

# NOTES:
# 1-99th is more lenient — keeps more signal but less outlier protection
# Compare the describe() output with the 5-95th cell above
# If std values are similar between both, the data does not have extreme outliers
# If std values differ significantly, extreme outliers are present and 5-95th is more appropriate

In [ ]:
dfx_1_99.head()

# Covariance Matrix

In [ ]:
# Covariance Matrix: 1-99th Percentile Clipping

# np.cov(): calculates how much each pair of channels vary together
# rowvar=False: tells numpy that each column is a channel (not each row)
# The result is a 12x12 matrix (every channel is compared against every other channel)
covariance_matrix_1_99 = np.cov(X_scaled_1_99, rowvar=False)

# Converting "covariance_matrix" to a Dataframe so the rows and columns have proper channel name labels
# Without this, it would just show numbers 0-11 instead of the channel element names
cov_df_1_99 = pd.DataFrame(
    covariance_matrix_1_99,
    index=channel_names_filtered,
    columns=channel_names_filtered
)

plt.figure(figsize=(12, 10))

# imshow(): this displays the matrix as a heatmap
# cmap='coolwarm': blue is negative correlation, white is no correlation and red is positive correlation
# vmin=-1, vmax=1: fixes the colour scale so it's always anchored at -1 and +1
im = plt.imshow(cov_df_1_99, cmap='coolwarm', vmin=-1, vmax=1)

# Adds colour bar on right hand side
plt.colorbar(im, label='Covariance')

# Labelling x and y axis with the channel names
plt.xticks(range(len(channel_names_filtered)), channel_names_filtered, rotation=45, ha='right')
plt.yticks(range(len(channel_names_filtered)), channel_names_filtered, rotation=45, ha='right')

# This nested for loop writes the actual number inside every single cell of the heatmap
for row in range(len(channel_names_filtered)):
    for col in range(len(channel_names_filtered)):
        plt.text(col, row, f"{cov_df_1_99.iloc[row, col]:.2f}",
                 ha='center', va='center', fontsize=7, color='black')

plt.title('Covariance Matrix — 1-99th Percentile Clipping')
plt.tight_layout()
plt.show()


#### OUTPUT EXPLAINED ####
# The covariance matrix is a mathematical summary of how all the channels relate to each other
# before PCA can find its principal components, it first needs to understand the structure of the data (which channels tend to move together across pixels, and which don't)
# The covariance matrix captures this. Every cell in the grid checks "when this element is high in a pixel, does that other element tend to be high or low too?"
# This is directly fed into the PCA, it's job is to find the directions in the data that capture the most variation
# These directions are calculated from the covariance matrix
# Sklearn computres the covariance matrix silently when you call "pca.fit_transform()". This matrix below just visualises what PCA is ging to find before you run PCA

# Diagonal: always 1.00, as every channel perfectly correlates with itself
# Red cells: positive correlation (close to +1). This means the two channels increase together in the same pixels. Example, potassium and sodium below shows 0.95, meaning wherever potassium is high in the tissue, sodium tends to be high too (biologically, this could mean they're co-localised in the same tissue structure)
# Blue cells: negative correlation (close to -1). This is where one channel is high, the other tends to be low. This could suggest two elements occupy different tissue compartments
# White (close to 0): no meaningful relationship beyween those two channels

# Eigendecomposition 

In [ ]:
# Eigendecomposition: 1-99th Percentile Clipping

# np.linalg.eig(): decomposes the covariance matrix into two things, eigen_values and eigen_vectors
# Eigen_values: a list of 12 numbers, one per component each value represents how much variance that component captures
# eigen_vectors: a 12x12 matrix where each column is one eigenvector (one PC), each value in the column is that channel's contribution/loading
eigen_values_1_99, eigen_vectors_1_99 = np.linalg.eig(covariance_matrix_1_99)

# The eigenvalues come out in no particular order
# np.argsort(): returns the indices that would sort the array low to high
# [::-1]: reverses that high to low (largest variance first)
# [:]: keeps all 12
sorted_key_1_99 = np.argsort(eigen_values_1_99)[::-1]

# Applying the sort order to both eigenvalues and eigenvectors, so that they stay matched up (PC1's value stays paired with PC1's vector)
eigen_values_1_99 = eigen_values_1_99[sorted_key_1_99]

# For eigenvectors we sort columns [:, sorted_key] not rows, as each component is stored as a column, not a row
eigen_vectors_1_99 = eigen_vectors_1_99[:, sorted_key_1_99]

## Print variance explained by each component ##
# Summing all eigenvalues gives the total variance in the dataset
total_variance_1_99 = np.sum(eigen_values_1_99)

# Dividing each eigenvalue by the total converts it to a percentage
# This tells you how much of the overall variation each PC accounts for
explained_eigen_1_99 = (eigen_values_1_99 / total_variance_1_99) * 100

# Printing each PC's individual and cumulative variance
# enumerate() gives both the index (j) and value (val) at the same time
# np.cumsum() calculates the running total 
print("1-99th Percentile: Variance explained per component (via eigenvalues):")
for j, val in enumerate(explained_eigen_1_99):
    print(f"  PC{j+1}: {val:.1f}%  |  Cumulative: {np.cumsum(explained_eigen_1_99)[j]:.1f}%")

# Fitting PCA 

In [ ]:
# Fitting PCA: 1-99th Percentile Clipping
pca_1_99 = PCA(n_components=len(channel_names_filtered))
X_pca_1_99 = pca_1_99.fit_transform(X_scaled_1_99)


# pca.fit_transform(): This function combines two steps into one:
# 1.) pca.fit(): learns the principal components from the data
# 2.) pca.transform(): projects every pixel onto those components

# By combining them, X_pca immediuatelt contains the coordinates of every pixel in PCA space
# This makes the model ready to be used directly in the scatter plot and loading plots, without needing a seperate transform step

# Scree Plot 

In [ ]:
# Scree Plot: 1-99th Percentile Clipping
plt.figure(figsize=(10, 6))

plt.scatter(x=[i+1 for i in range(len(pca_1_99.explained_variance_ratio_))],
            y=pca_1_99.explained_variance_ratio_,
            s=200, alpha=0.75, c='orange', edgecolor='k', label='Component Variance')

plt.plot([i+1 for i in range(len(pca_1_99.explained_variance_ratio_))],
         pca_1_99.explained_variance_ratio_,
         c='orange', linestyle='--', alpha=0.5)

plt.grid(True)
plt.title("Scree Plot — 1-99th Percentile Clipping\n", fontsize=25)
plt.xlabel("Principal Components", fontsize=15)
plt.ylabel("Ratio of Variance Explained", fontsize=15)
plt.xticks([i+1 for i in range(len(pca_1_99.explained_variance_ratio_))], fontsize=15)
plt.yticks(fontsize=15)
plt.show()

# Cumulative Variance Plot 

In [ ]:
# Cumulative Variance Plot: 1-99th Percentile Clipping

# Multiplying by 100 converts to a clean percentage
exp_var_cumul_1_99 = np.cumsum(pca_1_99.explained_variance_ratio_) * 100

# Creating the filled area plot
fig = px.area(
    x=range(1, exp_var_cumul_1_99.shape[0] + 1),
    y=exp_var_cumul_1_99,
    labels={"x": "Number of Principal Components", "y": "Cumulative Explained Variance (%)"},
    title="Cumulative Explained Variance — 1-99th Percentile Clipping"
)

# Adding dots so you can hover and see exact percentage values per component
fig.update_traces(mode='lines+markers')

# Forcing Y axis to 100% so the scale is always consistent
fig.update_layout(yaxis_range=[0, 105])

fig.show()


#### OUTPUT EXPLAINED ####
# This plot shows the running total of variance explained as you add more components
# Use this alongside the scree plot to decide how many PCs to keep
# A common threshold is 90-95% — the point where the curve flattens out
# For 12 channels, if 3-4 components reach 90%, the remaining channels
# are largely redundant and likely represent noise in the tissue signal

# PCA Scatter Matrix 

In [ ]:
# PCA Scatter Matrix: 1-99th Percentile Clipping

# Transforming the data to get PC coordinates
# This calculates exactly where each pixel sits in PCA space
# We already have X_pca from fit_transform() so we convert it to a DataFrame here
X_pca_df = pd.DataFrame(data=X_pca_1_99)

# Decide how many components you want to look at
# Setting this to 5 will show PC1 through to PC5 for example 
num_components = 4

# Creating the labels automatically
# Loops through the fitted pca model to grab the variance % for each PC
labels = {
    str(i): f"PC {i+1} ({var * 100:.1f}%)"
    for i, var in enumerate(pca_1_99.explained_variance_ratio_[:num_components])
}

# Subsampling — filtering to top 75% intensity pixels first
# This avoids sampling mostly background pixels which cause blob effects
# quantile(0.75) finds the intensity value that 75% of pixels fall below
# We only sample from pixels above that threshold
# Setting np.random.seed(42), locks the random number generator so it always picks the same 10,000 pixels every time this cell is run
total_intensity = df.sum(axis=1)
high_intensity_idx = total_intensity[total_intensity > total_intensity.quantile(0.75)].index
np.random.seed(42)
sample_idx = np.random.choice(high_intensity_idx, min(10000, len(high_intensity_idx)), replace=False)
X_pca_sample = X_pca_df.iloc[sample_idx].reset_index(drop=True)

# Subsampling the colour values to match the same pixels
# Colouring by Mg intensity — change index to explore other channels
color_channel_idx = 6
channel_name = channel_names_filtered[color_channel_idx]
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df.iloc[sample_idx, color_channel_idx].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

print(f"{channel_name} suggestion: {suggestion}")

# Always show linear scatter matrix
fig_linear = px.scatter_matrix(
    X_pca_sample,
    labels=labels,
    dimensions=range(num_components),
    color=color_values_linear,
    color_continuous_scale='hot'
)
fig_linear.update_traces(diagonal_visible=False, marker=dict(size=5, opacity=1))
fig_linear.update_layout(
    width=2000, height=2000,
    title=f"PCA Scatter Matrix — 1-99th Percentile Clipping (PC1 to PC{num_components}) — {channel_name} Linear Scale",
    coloraxis_colorbar=dict(title=channel_name)
)
fig_linear.show()

# Only show log scatter matrix if flagged as LOG or POSSIBLY LOG
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter_matrix(
        X_pca_sample,
        labels=labels,
        dimensions=range(num_components),
        color=color_values_log,
        color_continuous_scale='hot'
    )
    fig_log.update_traces(diagonal_visible=False, marker=dict(size=5, opacity=1))
    fig_log.update_layout(
        width=2000, height=2000,
        title=f"PCA Scatter Matrix — 1-99th Percentile Clipping (PC1 to PC{num_components}) — {channel_name} Log Scale (auto-flagged: {suggestion})",
        coloraxis_colorbar=dict(title=f"{channel_name} (log)")
    )
    fig_log.show()

In [ ]:
# PCA 2D Scatter Plot: 1-99th Percentile Clipping

# Change the PCs to better visualise specific combinations
# (0 = PC1, 1 = PC2, 2 = PC3, 3 = PC4 etc.)
X_AXIS = 0
Y_AXIS = 1

# Colouring by channel intensity — change index to explore other channels
color_channel_idx = 6
channel_name = channel_names_filtered[color_channel_idx]
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df.iloc[sample_idx, color_channel_idx].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

print(f"{channel_name} suggestion: {suggestion}")

# Always show linear plot
fig_linear = px.scatter(
    X_pca_sample, x=X_AXIS, y=Y_AXIS,
    color=color_values_linear,
    color_continuous_scale='hot',
    labels={
        str(X_AXIS): f"PC{X_AXIS+1} ({pca_1_99.explained_variance_ratio_[X_AXIS]*100:.1f}%)",  # ← pca_1_99
        str(Y_AXIS): f"PC{Y_AXIS+1} ({pca_1_99.explained_variance_ratio_[Y_AXIS]*100:.1f}%)"   # ← pca_1_99
    },
    title=f"PC{X_AXIS+1} vs PC{Y_AXIS+1}: {channel_name} — Linear Scale (1-99th Percentile Clipping)"
)
fig_linear.update_traces(marker=dict(size=8, opacity=1))
fig_linear.update_layout(coloraxis_colorbar=dict(title=channel_name), width=1000, height=800)
fig_linear.show()

# Only show log plot if flagged as LOG or POSSIBLY LOG
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter(
        X_pca_sample, x=X_AXIS, y=Y_AXIS,
        color=color_values_log,
        color_continuous_scale='hot',
        labels={
            str(X_AXIS): f"PC{X_AXIS+1} ({pca_1_99.explained_variance_ratio_[X_AXIS]*100:.1f}%)",  # ← pca_1_99
            str(Y_AXIS): f"PC{Y_AXIS+1} ({pca_1_99.explained_variance_ratio_[Y_AXIS]*100:.1f}%)"   # ← pca_1_99
        },
        title=f"PC{X_AXIS+1} vs PC{Y_AXIS+1}: {channel_name} — Log Scale (auto-flagged: {suggestion}) (1-99th Percentile Clipping)"
    )
    fig_log.update_traces(marker=dict(size=8, opacity=1))
    fig_log.update_layout(coloraxis_colorbar=dict(title=f"{channel_name} (log)"), width=1000, height=800)
    fig_log.show()


#### OUTPUT EXPLAINED ####
# This scatter plot shows how the pixels are distributed in PCA space using 1-99th percentile clipped and normalised data
# Each point is a pixel, coloured by the intensity of a specific channel
# The only difference from the main PCA scatter plot is that the input data had outliers clipped
# before normalisation — so the PCA axes may capture slightly different variance structure
# compared to the unclipped version, particularly if extreme hotspot pixels were driving PC directions
# Comparing this plot side by side with the main version reveals how sensitive the PCA structure
# is to outlier pixels — if the plots look similar, outliers had minimal effect on the analysis

In [ ]:
import plotly.express as px

# 3D PCA Scatter Plot: 1-99th Percentile Clipping
# Calculate the total variance explained by the first 3 components
total_var_3d_1_99 = pca_1_99.explained_variance_ratio_[:3].sum() * 100

# Colouring by channel intensity — change index to explore other channels
color_channel_idx = 2
channel_name = channel_names_filtered[color_channel_idx]
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df.iloc[sample_idx, color_channel_idx].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

print(f"{channel_name} suggestion: {suggestion}")

# Always show linear 3D plot
fig_linear = px.scatter_3d(
    X_pca_sample,
    x=0, y=1, z=2,
    color=color_values_linear,
    color_continuous_scale='hot',
    title=f'3D PCA Plot — {channel_name} Linear Scale ({total_var_3d_1_99:.2f}% variance explained) (1-99th Percentile Clipping)',
    labels={
        '0': f"PC1 ({pca_1_99.explained_variance_ratio_[0]*100:.1f}%)",  # ← pca_1_99
        '1': f"PC2 ({pca_1_99.explained_variance_ratio_[1]*100:.1f}%)",  # ← pca_1_99
        '2': f"PC3 ({pca_1_99.explained_variance_ratio_[2]*100:.1f}%)"   # ← pca_1_99
    }
)
fig_linear.update_traces(marker=dict(size=5, opacity=1))
fig_linear.update_layout(
    scene=dict(aspectmode='cube'),
    coloraxis_colorbar=dict(title=channel_name),
    width=1000, height=900
)
fig_linear.show()

# Only show log 3D plot if flagged as LOG or POSSIBLY LOG
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter_3d(
        X_pca_sample,
        x=0, y=1, z=2,
        color=color_values_log,
        color_continuous_scale='hot',
        title=f'3D PCA Plot — {channel_name} Log Scale (auto-flagged: {suggestion}) (1-99th Percentile Clipping)',
        labels={
            '0': f"PC1 ({pca_1_99.explained_variance_ratio_[0]*100:.1f}%)",  # ← pca_1_99
            '1': f"PC2 ({pca_1_99.explained_variance_ratio_[1]*100:.1f}%)",  # ← pca_1_99
            '2': f"PC3 ({pca_1_99.explained_variance_ratio_[2]*100:.1f}%)"   # ← pca_1_99
        }
    )
    fig_log.update_traces(marker=dict(size=5, opacity=1))
    fig_log.update_layout(
        scene=dict(aspectmode='cube'),
        coloraxis_colorbar=dict(title=f"{channel_name} (log)"),
        width=1000, height=900
    )
    fig_log.show()

# 2D Loading Plots 

In [ ]:
# Loading Plot: 1-99th Percentile Clipping

from matplotlib.lines import Line2D

# Settings
# Number of PCs to include in the loading matrix
# Adjust based on the scree plot elbow point
num_components = 4

loadings_1_99 = pca_1_99.components_

# Generating a distinct colour for each of the 12 channels
colours = plt.cm.tab20(np.linspace(0, 1, len(channel_names_filtered)))

fig, axes = plt.subplots(num_components, num_components, figsize=(20, 20), sharex=True, sharey=True)

for i in range(num_components):
    for j in range(num_components):
        ax = axes[i, j]
        if i == j:
            ax.axis('off')
            continue

        xs = loadings_1_99[j]
        ys = loadings_1_99[i]

        for k, name in enumerate(channel_names_filtered):
            current_colour = colours[k]

            # Line from origin to loading point
            ax.plot([0, xs[k]], [0, ys[k]], color=current_colour, alpha=0.8, zorder=2, linewidth=2)

            # Circle at the tip
            ax.scatter(xs[k], ys[k], s=500, color=current_colour, zorder=3, edgecolors='black')

            # Element name inside the circle
            ax.text(xs[k], ys[k], name.split(' ')[0],
                    ha='center', va='center', fontsize=6,
                    fontweight='bold', color='black', zorder=4)

        ax.set_xlim(-1.1, 1.1)
        ax.set_ylim(-1.1, 1.1)
        ax.axhline(0, color='black', linestyle='--', alpha=0.5, zorder=1)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, zorder=1)
        ax.grid(True, alpha=0.3, zorder=0)

        if i == num_components - 1:
            x_var = pca_1_99.explained_variance_ratio_[j] * 100
            ax.set_xlabel(f"PC{j+1}\n({x_var:.1f}%)", fontsize=16, weight='bold')

        if j == 0:
            y_var = pca_1_99.explained_variance_ratio_[i] * 100
            ax.set_ylabel(f"PC{i+1}\n({y_var:.1f}%)", fontsize=16, weight='bold')

plt.suptitle(f"Loading Matrix — 1-99th Percentile Clipping (PC1 to PC{num_components})", fontsize=30, y=0.92)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 2D Loading Plot: 1-99th Percentile Clipping
# The loading plot tells us which elements are driving each principal component
# Arrows pointing in the same direction = those elements are correlated
# Arrows pointing in opposite directions = those elements are inversely related
# Long arrows = strong influence on that PC, short arrows = weak influence

# Settings: Change these to plot different PC's
# (0 = PC1, 1 = PC2, 2 = PC3, 3 = PC4 etc.)
X_AXIS = 0
Y_AXIS = 1

# Getting the loading values from the 1-99th percentile clipped PCA model
loadings_1_99 = pca_1_99.components_  
xs = loadings_1_99[X_AXIS]
ys = loadings_1_99[Y_AXIS]

# Generating a distinct colour for each channel
colours = plt.cm.tab20(np.linspace(0, 1, len(channel_names_filtered)))

# Creating the Loading Plot
plt.figure(figsize=(10, 10))

for i, name in enumerate(channel_names_filtered):
    current_colour = colours[i]

    plt.scatter(xs[i], ys[i], s=200, color=current_colour, label=name)

    plt.arrow(
        0, 0,
        xs[i], ys[i],
        color=current_colour,
        head_width=0.02,
        length_includes_head=True
    )

plt.grid(True)
plt.xlim(-1, 1)
plt.ylim(-1, 1)

plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.axvline(0, color='black', linestyle='--', alpha=0.5)

plt.title("2D Loading Plot: 1-99th Percentile Clipping", fontsize=20)  # ← updated title
plt.xlabel(f"PC{X_AXIS+1} ({pca_1_99.explained_variance_ratio_[X_AXIS]*100:.1f}%)", fontsize=15)  # ← pca_1_99
plt.ylabel(f"PC{Y_AXIS+1} ({pca_1_99.explained_variance_ratio_[Y_AXIS]*100:.1f}%)", fontsize=15)  # ← pca_1_99

plt.legend(bbox_to_anchor=(1.04, 0.5), loc="center left", borderaxespad=0,
           title="Element Key", fontsize=12, title_fontsize=14)

plt.tight_layout()
plt.show()

# <font size="8">t-SNE</font>


# t-SNE implementation 

In [ ]:
# This is the number of PC's that I decided to keep based on the scree plot and cumulative variance plot
# This is important for t-SNE, UMAP and K-means
n_meaningful_components = 4 

In [ ]:
from sklearn.manifold import TSNE # imports the t-SNE algorithm from scikit-learn's manifold module. "Manifold" refers to the mathematical concept of finding lower-dimensional structure hidden in high-dimensional data.

# Running t-SNE on the top 4 PCA components
# Using PCA components as input rather than raw data speeds up t-SNE significantly and removes noise from less important channels
# perplexity controls how many neighbours each point considers — typically 5-50
# Higher perplexity = more global structure, lower = more local clusters
# max_iter = number of optimisation steps — 1000 is a safe minimum

# Creating the t-SNE model with these settings:
# n_components=2: reduce down to 2 dimensions so it can be plotted on an X/Y scatter plot
# perplexity=30: each point considers its 30 nearest neighbours when deciding where to position itself 
# max_iter=1000: t-SNE works by repeatedly adjusting point positions to better reflect similarities. 1000 rounds of adjustment is a safe minimum for stable results
# random_state=42: locks the random starting positions so you get same result every time you run it 
# verbose=1: prints progress updates while it runs so you can see it working 
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42, verbose=1)

# This is the input data. 
# Rather than feeding t-SNE all 10 raw channels, I am giving it only the top 4 PCA components of the high intensity pixels
# This is because t-SNE is very slow on high-dimensional data, PCA speeds this up massively 
# The 4 PCA components already capture the meaningful variation, we are not missing out on anything important 
# noise from weaker channels are already filtered out by PCA 
# tsne.fit_transform(...): runs the actual t-SNE algorithm. It starts with all the pixels placed randomly in 2D space, then iteratively moves them so that pixels that are similar in PCA space end up close together, and dissimilar pixels end up far apart. 
# X_tsne: This is the output, a 2D array of shape (112040, 2) where each row is a pixel and the two columns are its t-SNE coordinates. These coordinates have no physica meaning on its own, only the relative distances between points matter 
X_tsne = tsne.fit_transform(X_pca_1_99[high_intensity_idx, :n_meaningful_components])

print(f"t-SNE complete. Output shape: {X_tsne.shape}")

# t-SNE Visualisation

In [ ]:
# Colouring by channel intensity — change index to explore other channels
color_channel_idx = 7
channel_name = channel_names_filtered[color_channel_idx]
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df.iloc[high_intensity_idx, color_channel_idx].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

print(f"{channel_name} suggestion: {suggestion}")

# Always show linear t-SNE plot
fig_linear = px.scatter(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    color=color_values_linear,
    color_continuous_scale='hot',
    title=f't-SNE Plot — {channel_name} Linear Scale',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2'}
)
fig_linear.update_traces(marker=dict(size=5, opacity=0.6))
fig_linear.update_layout(
    coloraxis_colorbar=dict(title=channel_name),
    width=1000, height=800
)
fig_linear.show()

# Only show log t-SNE plot if flagged as LOG or POSSIBLY LOG
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter(
        x=X_tsne[:, 0],
        y=X_tsne[:, 1],
        color=color_values_log,
        color_continuous_scale='hot',
        title=f't-SNE Plot — {channel_name} Log Scale (auto-flagged: {suggestion})',
        labels={'x': 't-SNE 1', 'y': 't-SNE 2'}
    )
    fig_log.update_traces(marker=dict(size=5, opacity=0.6))
    fig_log.update_layout(
        coloraxis_colorbar=dict(title=f"{channel_name} (log)"),
        width=1000, height=800
    )
    fig_log.show()

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

n = len(channel_names_filtered)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for i, channel_name in enumerate(channel_names_filtered):
    
    # Get raw intensity values for this channel, for the high intensity pixels only
    color_vals = df.iloc[high_intensity_idx, i].reset_index(drop=True)
    
    # Use log scale if flagged, otherwise linear
    suggestion = scale_suggestions.get(channel_name, 'LINEAR')
    if suggestion in ['LOG', 'POSSIBLY LOG']:
        color_vals = np.log1p(color_vals)
        scale_label = '(log)'
    else:
        scale_label = ''

    sc = axes[i].scatter(
        X_tsne[:, 0], X_tsne[:, 1],
        c=color_vals,
        cmap='hot',
        s=0.5,
        rasterized=True
    )
    plt.colorbar(sc, ax=axes[i], shrink=0.7)
    axes[i].set_title(f'{channel_name} {scale_label}', fontsize=12)
    axes[i].axis('off')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('t-SNE — All Elements', fontsize=18, y=1.01)
plt.tight_layout()
plt.show()

# <font size="8">UMAP</font>

# Implementing UMAP

In [ ]:
%pip install umap-learn   # For some reason, running install with % instead of ! is necessary to get the package to import correctly in this notebook. Don't ask me why, I have no idea

In [ ]:
import umap

In [ ]:
# Running UMAP on the top 4 PCA components
# n_neighbors controls local vs global structure — typically 10-50
# Higher n_neighbors = more global structure preserved
# min_dist controls how tightly points are packed in the embedding
# Lower min_dist = tighter clusters, higher = more spread out

# Creating the UMAP model with these settings:
# n_components=2: Same as t-SNE, reduce down to 2 dimensions for plotting on an X/Y scatter plot
# n_neighbours=30: how many neighbouring pixels each point considers when learning the structure of the data. Low values (5-10) focus on very local structure and produce tighter, more seperated clusters. High values (50+) consider more global structure and produce a smoother embedding. 30 is good middle ground
# min_dist=0.1: This controls how tightly UMAP packs points together in the 2D space. Lower values (0.01-0.1) create tighter clusters, while higher values (0.5-0.99) produce a more spread out embedding. 0.1 is a common default that balances cluster separation with overall structure
# random_state=42: locks the random starting positions so you get same result every time you run it
# verbose=True: prints progress updates while it runs so you can see it working
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42, verbose=True)

# Same input as t-SNE, the top 4 PCA components of the high intensity pixels only 
# reducer.fit_transform(...): Runs the UMAP algorithm. Unlike t-SNE which only optimises point positions iteratively, UMAP actually learns a mathematical function that maps high dimensional data to 2D
# This is why UMAP is much faster than t-SNE on large datasets, because it doesn't need to repeatedly adjust point positions. Instead, it learns the underlying structure of the data and applies that function to get the 2D coordinates in one step
# X_umap: This is the output, a 2D array of shape (112040, 2) where each row is a pixel and the two columns are its UMAP coordinates. Like t-SNE, these coordinates have no physical meaning on their own, only the relative distances between points matter
X_umap = reducer.fit_transform(X_pca_1_99[high_intensity_idx, :n_meaningful_components])

print(f"UMAP complete. Output shape: {X_umap.shape}")

# UMAP Visualisation

In [ ]:
# Colouring by channel intensity — change index to explore other channels
color_channel_idx = 7
channel_name = channel_names_filtered[color_channel_idx]
suggestion = scale_suggestions.get(channel_name, 'LINEAR')

color_values_linear = df.iloc[high_intensity_idx, color_channel_idx].reset_index(drop=True)
color_values_log = np.log1p(color_values_linear)

print(f"{channel_name} suggestion: {suggestion}")

# Always show linear UMAP plot
fig_linear = px.scatter(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    color=color_values_linear,
    color_continuous_scale='hot',
    title=f'UMAP Plot — {channel_name} Linear Scale',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2'}
)
fig_linear.update_traces(marker=dict(size=5, opacity=0.6))
fig_linear.update_layout(
    coloraxis_colorbar=dict(title=channel_name),
    width=1000, height=800
)
fig_linear.show()

# Only show log UMAP plot if flagged as LOG or POSSIBLY LOG
if suggestion in ['LOG', 'POSSIBLY LOG']:
    fig_log = px.scatter(
        x=X_umap[:, 0],
        y=X_umap[:, 1],
        color=color_values_log,
        color_continuous_scale='hot',
        title=f'UMAP Plot — {channel_name} Log Scale (auto-flagged: {suggestion})',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2'}
    )
    fig_log.update_traces(marker=dict(size=5, opacity=0.6))
    fig_log.update_layout(
        coloraxis_colorbar=dict(title=f"{channel_name} (log)"),
        width=1000, height=800
    )
    fig_log.show()

In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for i, channel_name in enumerate(channel_names_filtered):

    color_vals = df.iloc[high_intensity_idx, i].reset_index(drop=True)

    suggestion = scale_suggestions.get(channel_name, 'LINEAR')
    if suggestion in ['LOG', 'POSSIBLY LOG']:
        color_vals = np.log1p(color_vals)
        scale_label = '(log)'
    else:
        scale_label = ''

    sc = axes[i].scatter(
        X_umap[:, 0], X_umap[:, 1],
        c=color_vals,
        cmap='hot',
        s=0.5,
        rasterized=True
    )
    plt.colorbar(sc, ax=axes[i], shrink=0.7)
    axes[i].set_title(f'{channel_name} {scale_label}', fontsize=12)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('UMAP — All Elements', fontsize=18, y=1.01)
plt.tight_layout()
plt.show()

# <font size="8">K-means</font>

# Elbow Plot 

In [ ]:
# Finding the optimal 'K' (Elbow Method) for 1-99th Percentile

# WCSS (Within-Cluster Sum of Squares) measures how tight the clusters are
wcss_1_99 = []


X_kmeans_1_99 = X_pca_1_99[high_intensity_idx, :n_meaningful_components]

# Testing 1 through 10 clusters
for i in range(1, 11):
    # random_state=42 ensures we get the same result every time we run the code
    kmeans_1_99 = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans_1_99.fit(X_kmeans_1_99)
    wcss_1_99.append(kmeans_1_99.inertia_)

# Plotting the Elbow Graph
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss_1_99, marker='o', linestyle='--', color='blue', markersize=8)
plt.title("Elbow Method — 1-99th Percentile Clipping\n", fontsize=20)
plt.xlabel("Number of Clusters (k)", fontsize=15)
plt.ylabel("WCSS (Cluster Variance)", fontsize=15)
plt.xticks(range(1, 11), fontsize=12)
plt.grid(True, alpha=0.5)
plt.show()

# Print all 10 WCSS scores
print("All WCSS scores (1-99th):", wcss_1_99)
# Print the WCSS for k=3 as a reference point
print(f"The exact WCSS for 3 clusters is: {wcss_1_99[2]}")

# OUTPUT EXPLAINED:
# Look for the elbow — the point where the line stops dropping steeply and flattens out
# For LA-ICP-MS tissue data, the elbow is often around k=3-5

# Applying K-means (5-95th and 1-99th percentile)

In [ ]:
# Applying K-Means — 1-99th Percentile Clipping
# Change n_clusters based on your elbow plot result
kmeans_1_99 = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=42)

# Fitting on the PCA coordinates of high intensity pixels only
# Using the same n_meaningful_components decided from the scree plot
X_kmeans_1_99 = X_pca_1_99[high_intensity_idx, :n_meaningful_components]

# Assigns a cluster number to every pixel
cluster_labels_1_99 = kmeans_1_99.fit_predict(X_kmeans_1_99)

# Convert to string so plotly assigns distinct colours instead of a gradient
cluster_labels_1_99 = cluster_labels_1_99.astype(str)

print("K-Means clustering complete (1-99th).")
print(f"Cluster distribution: {pd.Series(cluster_labels_1_99).value_counts().to_dict()}")

# Visualising K-means Clustering in PCA Space 

In [ ]:
# Visualising K-Means Clusters in PCA Space — 1-99th Percentile Clipping
fig = px.scatter(
    x=X_pca_1_99[high_intensity_idx, X_AXIS],
    y=X_pca_1_99[high_intensity_idx, Y_AXIS],
    color=cluster_labels_1_99,
    title=f"K-Means Clusters (1-99th) — PC{X_AXIS+1} vs PC{Y_AXIS+1}",
    labels={
        "x": f"PC{X_AXIS+1} ({pca_1_99.explained_variance_ratio_[X_AXIS]*100:.1f}%)",
        "y": f"PC{Y_AXIS+1} ({pca_1_99.explained_variance_ratio_[Y_AXIS]*100:.1f}%)",
        "color": "K-Means Cluster"
    },
    opacity=0.6
)

fig.update_traces(marker=dict(size=8, opacity=0.6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(width=1000, height=900)
fig.show()

# K-means Clustering on t-SNE

In [ ]:
fig = px.scatter(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    color=cluster_labels_1_99,
    title='K-Means Clusters on t-SNE Embedding',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2', 'color': 'K-Means Cluster'}
)
fig.update_traces(marker=dict(size=5, opacity=0.6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(width=1000, height=800)
fig.show()

# K-means Clustering on UMAP

In [ ]:
fig = px.scatter(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    color=cluster_labels_1_99,
    title='K-Means Clusters on UMAP Embedding',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'color': 'K-Means Cluster'}
)
fig.update_traces(marker=dict(size=5, opacity=0.6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(width=1000, height=800)
fig.show()

# Mapping Clusters to Original Image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# high_intensity_idx are row positions in df
# tissue_indices_final maps df rows back to pixel positions in the image
# So we index tissue_indices_final with high_intensity_idx to get the correct pixel positions
kmeans_pixel_positions = tissue_indices_final[high_intensity_idx]

# Sanity check
print(f"KMeans pixel positions: {len(kmeans_pixel_positions)}")
print(f"Cluster labels length: {len(cluster_labels_1_99)}")

# Building the Cluster Map
cluster_map = np.full((height, width), fill_value=-1, dtype=int)
rows = kmeans_pixel_positions // width
cols = kmeans_pixel_positions % width
cluster_map[rows, cols] = cluster_labels_1_99.astype(int)  

# Automatically detect number of clusters 
n_clusters = kmeans_1_99.n_clusters
print(f"Number of clusters detected: {n_clusters}")

cluster_colours = ['steelblue', 'red', 'limegreen', 'orange',
                   'purple', 'cyan', 'magenta', 'yellow',
                   'pink', 'teal', 'gold', 'white']

cmap_colours = ['black'] + cluster_colours[:n_clusters]
cmap = ListedColormap(cmap_colours)

legend_elements = [Patch(facecolor='black', label='Background (unsampled)')]
for i in range(n_clusters):
    legend_elements.append(Patch(facecolor=cluster_colours[i], label=f'Cluster {i}'))

for i, channel_name in enumerate(channel_names_filtered):
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    axes[0].imshow(cluster_map, cmap=cmap, vmin=-1, vmax=n_clusters - 1, origin='upper')
    axes[0].set_title(f'K-Means Cluster Map ({n_clusters} Clusters)', fontsize=14)
    axes[0].axis('off')
    axes[0].legend(handles=legend_elements, loc='lower right', fontsize=10)

    vmin_ch, vmax_ch = threshold_lookup.get(channel_name, (0, np.percentile(img_filtered[i], 99)))
    axes[1].imshow(img_filtered[i], cmap='inferno', vmin=vmin_ch, vmax=vmax_ch, origin='upper')
    axes[1].set_title(f'Channel: {channel_name}', fontsize=14)
    axes[1].axis('off')

    plt.suptitle(f'Spatial Cluster Map vs {channel_name}', fontsize=18)
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.cluster import KMeans
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt

n_clusters = kmeans_1_99.n_clusters

intensity_colours = ['steelblue', 'limegreen', 'tomato', 'orange',
                     'purple', 'cyan', 'magenta', 'yellow',
                     'pink', 'teal', 'gold', 'white']
cmap_colours = ['black'] + intensity_colours[:n_clusters]
cmap_fixed = ListedColormap(cmap_colours)

legend_elements = [Patch(facecolor='black', label='Background')]
for k in range(n_clusters):
    legend_elements.append(Patch(facecolor=intensity_colours[k], label=f'Intensity Tier {k}'))

# img_reshaped is (all tissue pixels, channels) — built from tissue_indices_final
# We use high_intensity_idx to match the same pixels K-Means was trained on
img_reshaped_tissue = img_filtered.reshape(n_channels, -1).T  # (all pixels, channels)

for i, channel_name in enumerate(channel_names_filtered):

    # Use tissue_indices_final[high_intensity_idx] to get correct pixel subset
    channel_data = img_reshaped_tissue[tissue_indices_final[high_intensity_idx], i].reshape(-1, 1)

    kmeans_channel = KMeans(n_clusters=n_clusters, init='k-means++', max_iter=300, n_init=10, random_state=42)
    raw_labels = kmeans_channel.fit_predict(channel_data)

    # Sort clusters by mean intensity so tier 0 = lowest, tier n = highest
    cluster_means = [channel_data[raw_labels == k].mean() for k in range(n_clusters)]
    sorted_clusters = np.argsort(cluster_means)
    label_map = np.zeros_like(raw_labels)
    for new_label, old_label in enumerate(sorted_clusters):
        label_map[raw_labels == old_label] = new_label

    # Use kmeans_pixel_positions (same positions used in previous cell)
    channel_cluster_map = np.full((height, width), fill_value=-1, dtype=int)
    kmeans_pixel_positions = tissue_indices_final[high_intensity_idx]
    rows = kmeans_pixel_positions // width
    cols = kmeans_pixel_positions % width
    channel_cluster_map[rows, cols] = label_map

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    axes[0].imshow(channel_cluster_map, cmap=cmap_fixed, vmin=-1, vmax=n_clusters - 1, origin='upper')
    axes[0].set_title(f'K-Means: {channel_name} ({n_clusters} Tiers)', fontsize=14)
    axes[0].axis('off')
    axes[0].legend(handles=legend_elements, loc='lower right', fontsize=10)

    vmin_ch, vmax_ch = threshold_lookup.get(channel_name, (0, np.percentile(img_filtered[i], 99)))
    axes[1].imshow(img_filtered[i], cmap='inferno', vmin=vmin_ch, vmax=vmax_ch, origin='upper')
    axes[1].set_title(f'Raw Channel: {channel_name}', fontsize=14)
    axes[1].axis('off')

    plt.suptitle(f'Per-Channel K-Means vs Raw Image: {channel_name}', fontsize=16)
    plt.tight_layout()
    plt.show()